(практикум от gigachain : https://vk.com/video-22522055_456246105)

На практике соберем простого агента на Python с **GigaChat API**, добавим инструменты, а затем переделаем его под стандарт [**Model Context Protocol**](https://modelcontextprotocol.io/specification/latest), позволяющего унифицировать работу с агентами.

## Установка необходимых библиотек
Для работы нашего кода, необходимо установить ряд библиотек. Перечень библиотек составлен из расчета на запуск в среде Google Colabaratory, где уже предустановлено большое количество библиотек.

Если вы будете запускать код в другой среде, возможно потребуется установка дополнительных библиотек.  

In [ ]:
!pip install -q langchain-gigachat rich langchain==1.0.0a9

Мы установили:
- `langchain-gigachat` - библиотека для взаимодействия с GigaChat API и интеграции с фреймворками LangChain и LangGraph
- `langchain` - фреймворк для разработки ИИ-агентов и LLM-приложений
- `rich` - библиотека для наглядного форматирования вывода в терминале

### Почему мы используем LangChain и LangGraph?
Чтобы не писать код агента с нуля, удобно прибегать к готовым фреймворкам.

Самым «живым» стеком для LLM-приложений является пара фреймворков [LangChain](https://github.com/langchain-ai/langchain)/[LangGraph](https://github.com/langchain-ai/langgraph): у них больше всех звезд на GitHub, активное сообщество и много реальных [кейсов промышленного внедрения](https://www.langchain.com/built-with-langgraph).

## Получение API-ключа для GigaChat
Чтобы получить доступ к GigaChat API, нужно зарегистрироваться в [личном кабинете Studio](https://developers.sber.ru/studio/login), создать проект GigaChat API и сгенерировать ключ авторизации. Подробности смотрите в [документации](https://developers.sber.ru/docs/ru/gigachat/quickstart/ind-create-project).



## Создаем диалогового ассистента на GigaChat
### Одиночный запрос к модели
Для начала рассмотрим, как происходит отправка одиночного запроса к LLM и получение ответа от неё. Понимание этого позволит дальше встроить обращения к LLM в приложения по обработке данных и перейти к более сложным сценариям, например, к диалоговому ассистенту.

В коде ниже выполним следующие действия:
* создадим клиент `GigaChat`, используя авторизационный ключ `auth`
* вызовем модель с помощью метода `invoke()` с текстом запроса;
* выведем на экран объект `AIMessage` **целиком**, чтобы увидеть и текст ответа, и метаданные (например, `token_usage`, `model_name`, `finish_reason`, `x_headers`)




In [ ]:
from rich import print
# from dotenv import load_dotenv, find_dotenv
# load_dotenv(find_dotenv())

In [ ]:

api_key= '...'


In [ ]:
from langchain_gigachat import GigaChat
llm = GigaChat(
    credentials=api_key,  
     
    model="GigaChat-2-Max", 
    profanity_check=False,
    top_p=0.9,                        
    timeout=120,
    scope="GIGACHAT_API_CORP",        
    verify_ssl_certs=False       
)

In [ ]:
llm.get_models()

In [ ]:

from rich.console import Console

console = Console()
# llm = GigaChat(credentials=auth, verify_ssl_certs=False)

response = llm.invoke("Привет! Объясни простыми словами, что такое искусственный интеллект.")
console.print(response)


Ответ модели во фреймворке LangChain представляется в виде объекта класса `AIMessage`. Он содержит сам текст ответа и метаданные о запросе/ответе.

* `content` — текст, который сгенерировала модель; именно его обычно показывают пользователю.
* `additional_kwargs` — доп. поля для особых режимов (например, tool/function calls); в простом кейсе пусто.
* `response_metadata` — метаданные от провайдера/клиента модели:

  * `token_usage`:

    * `prompt_tokens` — токены входа (промпта);
    * `completion_tokens` — токены в ответе;
    * `total_tokens` — сумма входа и ответа;
    * `precached_prompt_tokens` — часть входных токенов пришла из кэша (prompt caching).
  * `model_name` — точное имя/версия модели.
  * `x_headers`:

    * `x-request-id` — уникальный ID запроса (полезно для трассировки);
    * `x-session-id` — ID сессии (связывает сообщения одного диалога);
    * `x-client-id` — идентификатор клиента (часто `None` в демо).
  * `finish_reason` — причина остановки генерации (типично: `stop` — штатно; `length` — достигнут лимит длины).
* `id` — идентификатор самого сообщения (UUID на стороне LangChain).
* `usage_metadata` — сводные счётчики от LangChain:

  * `input_tokens` — входные токены;
  * `output_tokens` — токены ответа;
  * `total_tokens` — сумма;
  * `input_token_details`:

    * `cache_read` — сколько входных токенов прочитано из кэша (соотносится с `precached_prompt_tokens`).



### Диалоговый режим с историей сообщений

Перейдем от одиночных запросов к **диалогу** с памятью контекста.

* Используем `GigaChat` и классы `SystemMessage`/`HumanMessage`/`AIMessage` из LangChain: история (`chat_history`) хранит все реплики диалога с ролями.
* Ввод пользователя читается в цикле (`input(...)`). Пустая строка или `exit/quit` — выход.
* На каждый ход:

  * добавляем в `chat_history` сообщение пользователя: `HumanMessage(content=...)`
  * вызываем модель **на всей истории**: `llm.invoke(chat_history)` — модель видит предыдущие реплики и отвечает с учётом контекста
  * добавляем ответ модели в историю: `AIMessage(content=response.content)`
  * выводим текст ответа на экран



In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_gigachat import GigaChat

llm = GigaChat(
    credentials=api_key,  
     
    model="GigaChat-2-Max", 
    profanity_check=False,
    top_p=0.9,                        
    timeout=120,
    scope="GIGACHAT_API_CORP",        
    verify_ssl_certs=False       
)

# Создадим системный промпт модели: текст с описанием тона, стиля и правил
# Добавим его в переменную, которая будет хранить историю сообщений

chat_history = [
    SystemMessage(content=(
        "Ты — дружелюбный ассистент. Отвечай кратко и по делу. "
        "Если не уверен/не знаешь, скажи об этом."
    ))
]

print("💬 Чат с GigaChat запущен.")
print("Введите сообщение и нажмите Enter.")
print("Чтобы выйти — введите пустую строку или команду exit/quit.\n")

while True:
    user_text = input("👤 Вы: ").strip()
    if user_text.lower() in ["exit", "quit", ""]:
        print("✅ Чат завершён.")
        break

    chat_history.append(HumanMessage(content=user_text))
    response = llm.invoke(chat_history)
    chat_history.append(AIMessage(content=response.content))

    print("🤖 GigaChat:", response.content, "\n")


Попробуйте поэкспериментировать самостоятельно с разными вариантами системных промптов:

```
Ты — преподаватель, объясняющий сложные вещи простыми словами. Строй ответ по схеме: идея в одном-двух предложениях → короткий бытовой пример → один термин с пояснением. Не используй код, если о нём не попросили.
```

```
Ты — технический ассистент. Формат каждого ответа:
1) Контекст (1–2 предложения);
2) Решение (шаги списком, максимум 5 пунктов);
3) Пример (1–2 строки без кода, если код не запрошен).
Не раскрывай внутренние шаги модели.
```
```
Ты — ассистент, который даёт ответ и в конце добавляет строку «Итого: …» с одним выводом. Не используй код и не описывай внутренние процессы. Пиши коротко.
```


Чтобы управлять диалогом, чат-модели получают **список сообщений с ролями** — каждая роль имеет свое назначение в контексте:

* **system** — системные инструкции: тон, стиль, формат и ограничения ответа; обычно ставится в начале и действует на весь диалог
* **user** — запрос или данные от пользователя, на которые модель отвечает с учётом `system` и истории
* **assistant** — ответ модели; в агентных сценариях может содержать вызовы инструментов (tool calls); предыдущие ответы остаются в истории для контекста
* **tool** — результат внешнего инструмента, связанный с вызовом через `tool_call_id`; не для пользователя напрямую, а для следующего шага модели

В терминах LangChain роли [соответствуют классам](https://docs.langchain.com/oss/python/langchain/messages): `SystemMessage` (system), `HumanMessage` (user), `AIMessage` (assistant), `ToolMessage` (tool).


## Переход от диалогового ассистента к агенту

В реализованном нами диалоговом ассистенте модель при формировании ответа использует только контекст системного промпта и пользовательских сообщений, а также свои общие (ограниченные) знания. В историю диалога попадают лишь сообщения ролей `system`, `user` и `assistant`. Поэтому здесь **нет** `AIMessage` с `tool_calls` и **нет** `ToolMessage` — такие сообщения появляются уже в **агентной** архитектуре.

Дальше мы перейдём к агенту: модель сможет инициировать вызовы инструментов (tool calls), результаты будут приходить как `ToolMessage` и возвращаться в контекст для продолжения ответа. В качестве инструментов подключим функции на Python, чтобы обогащать диалог новыми фактическими данными.

### Реализация инструментов и их привязка к объекту GigaChat

Подготовим инструменты и модель для создания агента.

Для этого объявим объявим Python-функции, [обернув их в специальный декоратор](https://docs.langchain.com/oss/python/langchain/tools) `@tool`.

Пусть у нас будет два инструмента:

* `get_weather(city)` — берёт текущее состояние погоды из публичного API `wttr.in` и возвращает короткую строку (подробнее см. [репозиторий](https://github.com/chubin/wttr.in));
* `get_air_quality(city)` — находит координаты города через Open-Meteo Geocoding и получает PM2.5/PM10 из Open-Meteо Air Quality.
  
Оба инструмента работают без ключей, синхронно через библиотеку `requests`, с таймаутами и простой обработкой ошибок (`"Ошибка: …"`).

Далее инициализируем `GigaChat` и собираем список `tools`, куда помещаем наши инструменты.

#### Несколько слов о параметрах PM2.5/PM10

Качество воздуха обычно оценивают по концентрациям взвешенных частиц (PM) и некоторых газов (озон O₃, диоксид азота NO₂, диоксид серы SO₂, угарный газ CO). Чтобы упростить, будем запрашивать только концентрацию частиц. Для частиц чаще всего смотрят PM2.5 (очень мелкая пыль до 2,5 мкм) и PM10 (до 10 мкм). Единицы измерения — мкг/м³: чем меньше число, тем чище воздух. Мы будем получать почасовые значения, которые можно принять за мгновенные.

Можете самостоятельно поэкспериментировать с созданием инструментов определения качества воздуха, подобрав другие параметры через запросы к [Open-Meteо](https://open-meteo.com/en/docs/air-quality-api).  

In [ ]:
from langchain.tools import tool
from langchain_gigachat import GigaChat
import requests

# --- Инструменты ---
@tool
def get_weather(city: str) -> str:
    """Получение текущей температуры через API wttr.in.
    Args:
        city (str): Название города (англ.)
    Returns:
        str: 'ясно, +20 °C' или сообщение об ошибке
    """
    url = f"https://wttr.in/{city}?format=j1"
    try:
        data = requests.get(url, timeout=60).json()
        temp_c = data["current_condition"][0]["temp_C"]
        desc   = data["current_condition"][0]["weatherDesc"][0]["value"]
        return f"{desc.lower()}, {temp_c} °C"
    except Exception as e:
        return f"Ошибка: {e}"


@tool
def get_air_quality(city: str) -> str:
    """Возвращает PM2.5 / PM10 для заданного города.
    Args:
        city (str): Название города (англ., с большой буквы)
    Returns:
        str: 'PM2.5 = …, PM10 = …' или сообщение об ошибке
    """
    try:
        # Определяем координаты города
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en"
        geo = requests.get(geo_url, timeout=60).json()
        if not geo.get("results"):
            return "Город не найден"

        lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]

        # Качество воздуха
        aq_url = (
            "https://air-quality-api.open-meteo.com/v1/air-quality"
            f"?latitude={lat}&longitude={lon}&hourly=pm10,pm2_5&forecast_days=1"
        )
        aq = requests.get(aq_url, timeout=60).json()
        pm25, pm10 = aq["hourly"]["pm2_5"][0], aq["hourly"]["pm10"][0]
        return f"PM2.5 = {pm25} µg/m³, PM10 = {pm10} µg/m³"
    except Exception as e:
        return f"Ошибка: {e}"




In [ ]:

import ssl
import httpx
from gigachat import GigaChat
from gigachat.settings import Settings

settings = Settings()
settings.verify_ssl_certs = False
# --- LLM ---

llm = GigaChat(
    credentials=api_key,  
     
    model="GigaChat-2-Max", 
    profanity_check=False,
                       
    timeout=120,
    scope="GIGACHAT_API_CORP",        
    verify_ssl_certs=False       
)

tools = [get_weather, get_air_quality]

print("✅ Тулы и LLM готовы")

Обратите внимание на описание функций в виде `docstring`. Если в обычных функциях эти описания составляются для разработчика и не влияют на работу кода, то с агентами это не так. Дело в том, что описание функции — это то, что модель видит в контексте: по нему она понимает назначение, ограничения и формат аргументов/ответа. Делайте `docstring` лаконичным и предметным. Чем более понятным будет описание, тем лучше будет работать ваш агент.



In [ ]:
from rich.table import Table 

In [ ]:

# Получаем список доступных моделей
response =llm.get_models()
console.print("[bold green]Доступные модели:[/bold green]")

table = Table(title="Модели GigaChat")
table.add_column("ID модели", style="cyan")
table.add_column("Поддержка функций", style="green")

for model in response.data:
    # У объектов Model обычно есть атрибут 'name' или 'model_name'
    model_name = getattr(model, 'name', None) or getattr(model, 'model_name', None) or str(model)
    
    # Проверяем поддержку функций
    supports_functions = "✅ Да" if "pro" in model_name.lower() or "max" in model_name.lower() else "❌ Нет"
    table.add_row(model_name, supports_functions)

console.print(table)

In [ ]:
llm_with_tools = llm.bind_tools(tools)
console.rule("[bold]Конфигурация вызова с tools")
console.print(llm_with_tools)

question = "Какая сейчас погода в Нижнем Новгороде?"
resp = llm_with_tools.invoke([HumanMessage(content=question)])
console.rule("[bold]Ответ модели (tool_call)")
console.print(resp)

 В ячейке кода выше мы видим объект `RunnableBinding`. В LangChain различные абстрации `Runnable` служат в качестве интерфейсов для объектов, которые можно запустить. Это могут быть модели, различные цепочки и т.д. При запуске `RunnableBinding` к запросу пользователя добавляются параметры из списка инструментов.


 Как видим, в `AIMessage()` поле `content` пусто, зато `finish_reason='function_call'`. Это  значит, что модель решила вызвать инструмент, а `tool_calls` показывает, какой именно инструмент и с какими аргументами нужно выполнить (здесь — `get_weather` с `city = "Нижний Новгород"`). В `response_metadata` — служебные метаданные (модель, токены, идентификаторы запроса/сессии), в `usage_metadata` — свод по токенам.

 Следующий шаг агентного цикла: выполнить вызов инструмента, вернуть результат как `ToolMessage`, после чего модель сформирует следующий `AIMessage` с новым запросом к инструментам или заполнит поле `content` уже с финальным текстовым ответом для пользователя.

In [ ]:
import langchain

In [ ]:
print(langchain.__version__)

In [ ]:
from langchain.agents import create_agent


agent_no_memory = create_agent(llm, tools)
agent_no_memory

Представить агента в виде схемы можно как показано на диаграмме выше. Узел **agent** означает вызов модели, узел **tools** — выполнение подключённых инструментов; обратная стрелка означает, что после `ToolMessage` цикл повторяется, пока модель не вернёт `AIMessage` **без** `tool_calls` (финальный ответ).

Теперь запустим получившегося агента в интерактивном цикле: на каждом вводе агент сам решает, какие инструменты звать, а в мы увидим последовательность шагов (Human → AI/tool_calls → Tool → AI).


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_gigachat import GigaChat
from langchain_core.tools import tool
from rich.console import Console
from rich.markdown import Markdown

console = Console()

# 1. Инициализация модели (обязательно используйте GigaChat-2-Pro или GigaChat-2-Max!)
llm = GigaChat(
    credentials=api_key,  # замените на ваш ключ
    verify_ssl_certs=False,
    scope="GIGACHAT_API_CORP",
    model="GigaChat-2-Pro",  # или "GigaChat-2-Max" — только они поддерживают инструменты
    temperature=0.7
)

# 2. Определяем инструмент (функцию) для погоды
@tool
def get_weather(city: str) -> str:
    """Получает текущую погоду в указанном городе"""
    # Здесь должен быть реальный API запрос к сервису погоды
    # Для примера возвращаем тестовые данные
    weather_data = {
        "Москва": "+5°C, облачно",
        "Санкт-Петербург": "+2°C, дождливо", 
        "Нижний Новгород": "-3°C, снег",
        "Питер": "+2°C, дождливо"
    }
    return weather_data.get(city, f"+10°C, солнечно в {city}")

# 3. Создаем агента с инструментами
agent = create_react_agent(
    model=llm,
    tools=[get_weather]
)

# 4. Функция для общения с агентом
def Каchat_with_agent():
    print("💬 Агент готов к работе. Пустая строка — выход.\n")
    
    while True:
        user_text = input("👤 Вы: ").strip()
        if not user_text:
            print("✅ Диалог завершён.")
            break
        
        # Запускаем агента
        response = agent.invoke({
            "messages": [("user", user_text)]
        })
        
        # Получаем последний ответ
        last_message = response["messages"][-1]
        
        print("\n🤖 Агент:\n")
        print(last_message.content)
        print("\n" + "-"*50 + "\n")

# Запускаем чат
chat_with_agent()

Давайте далее посмотрим как агентный цикл выглядит на практике.

## Реализация простого ReAct-агента

В качестве агента мы возьмем простой вариант с максимальной свободой действий, который называют ReAct-агентом.

ReAct — это паттерн, в котором модель чередует размышления, действия и наблюдения, вызывая внешние инструменты для решения многошаговых задач. При этом модель сама определяет на каждом шаге, какое действие предпринять.

Для создания агента мы обернём наши абстрации в виде инструментов и клиента для обращения к GigaChat API, в готовую реализацию из фреймворка LangGraph — `create_agent` из `langchain.agents`.

Этот конструктор сразу собирает минимальный ReAct-цикл как граф состояний: подключает модель и инструменты, обрабатывает структурные `tool_calls/ToolMessage` и выполняет шаги, определенные моделью.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_gigachat import GigaChat
from langchain_core.tools import tool
from rich.console import Console
from rich.markdown import Markdown

console = Console()

# 1. Инициализация модели (обязательно используйте Pro или Max версию!)
llm = GigaChat(
    credentials=api_key,  # замените на ваш ключ
    verify_ssl_certs=False,
    scope="GIGACHAT_API_CORP",
    model="GigaChat-2-Pro",  # или "GigaChat-2-Max" — только они поддерживают инструменты
    temperature=0.7
)

# 2. Определяем инструмент для погоды
@tool
def get_weather(city: str) -> str:
    """Получает текущую погоду в указанном городе"""
    # Здесь должен быть реальный API запрос
    weather_data = {
        "Москва": "+5°C, облачно",
        "Санкт-Петербург": "+2°C, дождливо", 
        "Нижний Новгород": "-3°C, снег",
        "Питер": "+2°C, дождливо"
    }
    return weather_data.get(city, f"+10°C, солнечно в {city}")

# 3. Создаем чекпойнтер для памяти (используем InMemorySaver)
checkpointer = InMemorySaver()  # ✅ Правильный класс

# 4. Создаем агента с памятью
agent_with_memory = create_react_agent(
    model=llm,
    tools=[get_weather],
    checkpointer=checkpointer  # Добавляем память
)

# 5. Функция для общения с агентом
def chat_with_agent():
    print("💬 Агент с памятью готов к работе. Пустая строка — выход.\n")
    
    # Создаем уникальный ID для сессии
    config = {"configurable": {"thread_id": "user_session_1"}}
    
    while True:
        user_text = input("👤 Вы: ").strip()
        if not user_text:
            print("✅ Диалог завершён.")
            break
        
        # Запускаем агента с конфигурацией памяти
        response = agent_with_memory.invoke(
            {"messages": [("user", user_text)]},
            config=config  # Важно: передаем config для сохранения памяти
        )
        
        # Получаем последний ответ
        last_message = response["messages"][-1]
        
        print("\n🤖 Агент:\n")
        print(last_message.content)
        print("\n" + "-"*50 + "\n")

# Запускаем чат
chat_with_agent()

Как видим по диалогу, агент теперь хранит общий контекст в рамках одного `thread_id`. Если хотите еще углубиться в вопросы управления памятью ReAct-агента, то посмотрите [документацию](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent-manage-message-history/).